In [788]:
import os
import pandas as pd
import json
import logging
from tqdm import tqdm

# Configuration du logging
logging.basicConfig(level=logging.ERROR, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('data_loader')

# Chemin vers le dossier data_f1
data_path = os.path.join(os.path.expanduser("~"), "Desktop", "data_f1")

# Structure des données JSON
with open(os.path.join(data_path, "f1_data_structure.json"), "r") as f:
    data_structure = json.load(f)

In [789]:
def load_drivers(driver_path):
    """Charge les données des pilotes."""
    df = pd.read_csv(driver_path)
    return df[['driver_number', 'session_key', 'full_name', 'team_name']].copy()

def load_sessions(session_path):
    """Charge les données des sessions."""
    df = pd.read_csv(session_path)
    return df[['session_key', 'meeting_key', 'session_name', 'session_type', 'year', 'circuit_key']].copy()

def load_intervals(intervals_path):
    """Charge les données des intervalles."""
    df = pd.read_csv(intervals_path)
    return df[['driver_number', 'date', 'interval', 'gap_to_leader']].copy()

def load_laps(laps_path):
    """Charge les données des tours."""
    df = pd.read_csv(laps_path)
    return df[['driver_number', 'session_key', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'st_speed']].copy()

def load_pit(pit_path):
    """Charge les données des arrêts aux stands."""
    df = pd.read_csv(pit_path)
    return df[['driver_number', 'session_key', 'lap_number', 'pit_duration']].copy()

def load_position(position_path):
    """Charge les données de position."""
    df = pd.read_csv(position_path)
    return df[['driver_number', 'date', 'position']].copy()

def load_race_control(race_control_path):
    """Charge les données de race control."""
    df = pd.read_csv(race_control_path)
    return df[['date', 'category', 'driver_number', 'flag', 'scope', 'sector']].copy()

def load_stint(stint_path):
    """Charge les données des relais."""
    df = pd.read_csv(stint_path)
    return df[['driver_number', 'session_key', 'lap_start', 'compound', 'stint_number', 'tyre_age_at_start', 'lap_end']].copy()

def load_weather(weather_path):
    """Charge les données météorologiques."""
    df = pd.read_csv(weather_path)
    return df[['date', 'air_temperature', 'humidity', 'pressure', 'rainfall', 'track_temperature', 'wind_direction', 'wind_speed']].copy()

In [ ]:
# Initialisation des listes pour chaque table
drivers_list = []
sessions_list = []
intervals_list = []
laps_list = []
pit_list = []
position_list = []
race_control_list = []
stint_list = []
weather_list = []

# Compteur total de fichiers à charger
total_files = 0
for year, meetings in data_structure.items():
    for meeting_key, meeting_data in meetings.items():
        for session_key, session_data in meeting_data.get("sessions", {}).items():
            total_files += len(session_data.get("common_files", []))
            for driver_number, driver_data in session_data.get("drivers", {}).items():
                total_files += len(driver_data.get("files", []))

# Boucle principale de chargement des données
with tqdm(total=total_files, desc="Chargement des fichiers", unit="file") as pbar:
    for year, meetings in data_structure.items():
        for meeting_key, meeting_data in meetings.items():
            circuit_key = meeting_data.get("circuit_key")  # Récupérer circuit_key ici
            for session_key, session_data in meeting_data.get("sessions", {}).items():
                session_path = os.path.join(data_path, year, meeting_key, session_key, "sessions.csv")
                if os.path.exists(session_path):
                    try:
                        sessions_list.append(load_sessions(session_path))
                    except Exception as e:
                        logger.error(f"Erreur lors du chargement de {session_path}: {e}")
                    pbar.update(1)
                else:
                    logger.error(f"Fichier session manquant: {session_path}")

                for driver_number, driver_data in session_data.get("drivers", {}).items():
                    driver_path = os.path.join(data_path, year, meeting_key, session_key, driver_number, "drivers.csv")
                    if os.path.exists(driver_path):
                        try:
                            drivers_list.append(load_drivers(driver_path))
                        except Exception as e:
                            logger.error(f"Erreur lors du chargement de {driver_path}: {e}")
                        pbar.update(1)
                    else:
                        logger.error(f"Fichier drivers manquant: {driver_path}")

                    # ... (chargement des autres fichiers pour chaque pilote : intervals, laps, pit, position, stint)
                    intervals_path = os.path.join(data_path, year, meeting_key, session_key, driver_number, "intervals.csv")
                    if os.path.exists(intervals_path):
                        try:
                            intervals_list.append(load_intervals(intervals_path))
                        except Exception as e:
                            logger.error(f"Erreur lors du chargement de {intervals_path}: {e}")
                        pbar.update(1)

                    laps_path = os.path.join(data_path, year, meeting_key, session_key, driver_number, "laps.csv")
                    if os.path.exists(laps_path):
                        try:
                            laps_list.append(load_laps(laps_path))
                        except Exception as e:
                            logger.error(f"Erreur lors du chargement de {laps_path}: {e}")
                        pbar.update(1)

                    pit_path = os.path.join(data_path, year, meeting_key, session_key, driver_number, "pit.csv")
                    if os.path.exists(pit_path):
                        try:
                            pit_list.append(load_pit(pit_path))
                        except Exception as e:
                            logger.error(f"Erreur lors du chargement de {pit_path}: {e}")
                        pbar.update(1)

                    position_path = os.path.join(data_path, year, meeting_key, session_key, driver_number, "position.csv")
                    if os.path.exists(position_path):
                        try:
                            position_list.append(load_position(position_path))
                        except Exception as e:
                            logger.error(f"Erreur lors du chargement de {position_path}: {e}")
                        pbar.update(1)

                    stint_path = os.path.join(data_path, year, meeting_key, session_key, driver_number, "stints.csv")
                    if os.path.exists(stint_path):
                        try:
                            stint_list.append(load_stint(stint_path))
                        except Exception as e:
                            logger.error(f"Erreur lors du chargement de {stint_path}: {e}")
                        pbar.update(1)

                # Chargement des fichiers communs à chaque session
                race_control_path = os.path.join(data_path, year, meeting_key, session_key, "race_control.csv")
                if os.path.exists(race_control_path):
                    try:
                        race_control_list.append(load_race_control(race_control_path))
                    except Exception as e:
                        logger.error(f"Erreur lors du chargement de {race_control_path}: {e}")
                    pbar.update(1)

                weather_path = os.path.join(data_path, year, meeting_key, session_key, "weather.csv")
                if os.path.exists(weather_path):
                    try:
                        weather_list.append(load_weather(weather_path))
                    except Exception as e:
                        logger.error(f"Erreur lors du chargement de {weather_path}: {e}")
                    pbar.update(1)

# Concaténation des données dans des DataFrames pandas
drivers_df = pd.concat(drivers_list, ignore_index=True)
sessions_df = pd.concat(sessions_list, ignore_index=True)
intervals_df = pd.concat(intervals_list, ignore_index=True)
laps_df = pd.concat(laps_list, ignore_index=True)
pit_df = pd.concat(pit_list, ignore_index=True)
position_df = pd.concat(position_list, ignore_index=True)
race_control_df = pd.concat(race_control_list, ignore_index=True)
stint_df = pd.concat(stint_list, ignore_index=True)
weather_df = pd.concat(weather_list, ignore_index=True)

In [ ]:
# Affichage des premières lignes de chaque table pour vérification
print("Drivers:")
display(drivers_df.head())

print("\nSessions:")
display(sessions_df.head())

print("\nIntervals:")
display(intervals_df.head())

print("\nLaps:")
display(laps_df.head())

print("\nPit:")
display(pit_df.head())

print("\nPosition:")
display(position_df.head())

print("\nRace Control:")
display(race_control_df.head())

print("\nStint:")
display(stint_df.head())

print("\nWeather:")
display(weather_df.head())

In [ ]:
def describe_with_additional_info(df, df_name):
    """Affiche des statistiques descriptives, le nombre de valeurs manquantes et le type de données pour chaque colonne d'un DataFrame."""
    print(f"Statistiques descriptives pour {df_name}:")
    
    # Gestion de datetime_is_numeric pour les versions antérieures à pandas 1.1.0
    try:
        display(df.describe(include='all', datetime_is_numeric=True))
    except TypeError:
        display(df.describe(include='all'))

    print(f"\nInformations supplémentaires pour {df_name}:")
    info_df = pd.DataFrame({
        'Colonne': df.columns,
        'Valeurs manquantes': df.isnull().sum(),
        'Pourcentage de valeurs manquantes': (df.isnull().sum() / len(df)) * 100,
        'Type de données': df.dtypes
    })
    display(info_df)
    print("\n" + "-"*50 + "\n")

# Statistiques pour chaque table
describe_with_additional_info(drivers_df, "Drivers")
describe_with_additional_info(sessions_df, "Sessions")
describe_with_additional_info(intervals_df, "Intervals")
describe_with_additional_info(laps_df, "Laps")
describe_with_additional_info(pit_df, "Pit")
describe_with_additional_info(position_df, "Position")
describe_with_additional_info(race_control_df, "Race Control")
describe_with_additional_info(stint_df, "Stint")
describe_with_additional_info(weather_df, "Weather")

## Analyse des Données de F1

Ce document présente une analyse des données de Formule 1 extraites de plusieurs fichiers CSV. Les données sont structurées en huit tables distinctes, chacune contenant des informations spécifiques sur les pilotes, les sessions, les intervalles de temps, les tours, les arrêts aux stands, les positions, les contrôles de course, les relais et les conditions météorologiques.

### Structure des Tables

Voici un résumé de la structure de chaque table :

*   **Drivers** (Pilotes) :
    *   `driver_number` (INT, PK) : Numéro unique du pilote.
    *   `session_key` (INT, PK) : Clé unique de la session.
    *   `full_name` (VARCHAR(50)) : Nom complet du pilote.
    *   `team_name` (VARCHAR(50)) : Nom de l'équipe du pilote.

*   **Sessions** :
    *   `session_key` (INT, PK) : Clé unique de la session.
    *   `session_name` (VARCHAR(50)) : Nom de la session (par exemple, "Practice 1").
    *   `session_type` (VARCHAR(50)) : Type de session (par exemple, "Practice", "Qualifying", "Race").
    *   `year` (INT) : Année de la session.
    *   `circuit_key` (INT) : Clé unique du circuit.

*   **Intervals** (Intervalles) :
    *   `driver_number` (INT, PK) : Numéro unique du pilote.
    *   `date` (TIMESTAMP, PK) : Date et heure de l'enregistrement.
    *   `interval` (FLOAT) : Intervalle de temps avec le pilote précédent.
    *   `gap_to_leader` (FLOAT) : Écart de temps avec le leader.

*   **Laps** (Tours) :
    *   `driver_number` (INT, PK) : Numéro unique du pilote.
    *   `session_key` (INT, PK) : Clé unique de la session.
    *   `lap_number` (INT, PK) : Numéro du tour.
    *   `date_start` (TIMESTAMP) : Date et heure de début du tour.
    *   `duration_sector_1` (FLOAT) : Durée du secteur 1.
    *   `duration_sector_2` (FLOAT) : Durée du secteur 2.
    *   `duration_sector_3` (FLOAT) : Durée du secteur 3.
    *   `i1_speed` (INT) : Vitesse au point intermédiaire 1.
    *   `i2_speed` (INT) : Vitesse au point intermédiaire 2.
    *   `is_pit_out_lap` (BOOLEAN) : Indique si le tour est un tour de sortie des stands.
    *   `lap_duration` (FLOAT) : Durée totale du tour.
    *   `st_speed` (FLOAT) : Vitesse au speed trap.

*   **Pit** (Arrêts aux stands) :
    *   `driver_number` (INT, PK) : Numéro unique du pilote.
    *   `session_key` (INT, PK) : Clé unique de la session.
    *   `lap_number` (INT, PK) : Numéro du tour.
    *   `pit_duration` (FLOAT) : Durée de l'arrêt aux stands.

*   **Position** :
    *   `driver_number` (INT, PK) : Numéro unique du pilote.
    *   `date` (TIMESTAMP, PK) : Date et heure de l'enregistrement.
    *   `position` (INT) : Position du pilote.

*   **Race_control** (Contrôle de course) :
    *   `date` (TIMESTAMP, PK) : Date et heure de l'événement.
    *   `category` (VARCHAR(50)) : Catégorie de l'événement (par exemple, "Flag", "Other").
    *   `driver_number` (INT) : Numéro unique du pilote (si applicable).
    *   `flag` (VARCHAR(50)) : Type de drapeau (par exemple, "GREEN", "RED").
    *   `scope` (VARCHAR(50)) : Portée de l'événement (par exemple, "Track", "Sector").
    *   `sector` (INT) : Secteur concerné (si applicable).

*   **Stint** (Relais) :
    *   `driver_number` (INT, PK) : Numéro unique du pilote.
    *   `session_key` (INT, PK) : Clé unique de la session.
    *   `lap_start` (INT, PK) : Numéro du tour de début du relais.
    *   `compound` (VARCHAR(50)) : Type de gomme de pneu utilisée.
    *   `stint_number` (INT) : Numéro du relais.
    *   `tyre_age_at_start` (INT) : Âge du pneu au début du relais (en nombre de tours).
    *   `lap_end` (INT) : Numéro du tour de fin du relais.

*   **Weather** (Météo) :
    *   `date` (TIMESTAMP, PK) : Date et heure de l'enregistrement.
    *   `air_temperature` (FLOAT) : Température de l'air.
    *   `humidity` (INT) : Humidité relative.
    *   `pressure` (FLOAT) : Pression atmosphérique.
    *   `rainfall` (INT) : Précipitations (0 ou 1).
    *   `track_temperature` (FLOAT) : Température de la piste.
    *   `wind_direction` (INT) : Direction du vent.
    *   `wind_speed` (FLOAT) : Vitesse du vent.

### Analyse des Résultats

#### Drivers (Pilotes)

*   **Données manquantes :** La colonne `team_name` présente un nombre significatif de valeurs manquantes (21.8%). Il faudra envisager une stratégie pour les imputer ou les supprimer si nécessaire pour certaines analyses.
*   **Diversité des pilotes et des équipes :** On observe 53 pilotes uniques et 12 équipes uniques, avec une prédominance de l'équipe McLaren dans les données fournies.
*   **Distribution des données :** La moyenne des numéros de pilotes est de 28.5, ce qui suggère une distribution relativement large des numéros de pilotes dans l'ensemble des données.

#### Sessions

*   **Types de sessions :** Les données couvrent 8 types de sessions différents, principalement des "Practice" (essais).
*   **Période couverte :** Les données s'étendent sur les années 2023 et 2024, avec une majorité de sessions en 2024.
*   **Circuits :** Les données proviennent de plusieurs circuits, avec une clé de circuit maximale de 152.

#### Intervals (Intervalles)

*   **Valeurs manquantes :** Les colonnes `interval` et `gap_to_leader` présentent respectivement 1.4% et 8.9% de valeurs manquantes. Ces données sont cruciales pour l'analyse des performances en course, et une imputation pourrait être envisagée.
*   **Données aberrantes :** La valeur maximale de `gap_to_leader` semble inhabituellement élevée, ce qui pourrait indiquer des erreurs de saisie ou des événements exceptionnels en course.

#### Laps (Tours)

*   **Valeurs manquantes :** Plusieurs colonnes présentent des valeurs manquantes, notamment `duration_sector_1`, `i1_speed`, `lap_duration` et `st_speed`. L'imputation de ces valeurs est essentielle pour une analyse complète des performances au tour.
*   **Données aberrantes :** Des valeurs extrêmes sont observées dans `duration_sector_1`, `duration_sector_3`, et `lap_duration`, ce qui peut être dû à des erreurs d'enregistrement ou à des événements inhabituels (tours d'entrée/sortie des stands, drapeaux rouges, etc.).
*   **Vitesses :** Les vitesses moyennes aux points intermédiaires et au speed trap sont cohérentes avec les attentes pour des courses de F1.

#### Pit (Arrêts aux stands)

*   **Valeurs manquantes :** La colonne `pit_duration` présente 16.8% de valeurs manquantes.
*   **Durée des arrêts :** La durée moyenne des arrêts aux stands est d'environ 6 minutes, avec un écart type élevé, ce qui suggère une grande variabilité dans les durées d'arrêt.

#### Position

*   **Pas de valeurs manquantes :** C'est un point positif, car les données de position sont complètes.
*   **Distribution des positions :** La position la plus fréquente est la 8ème place, ce qui est cohérent avec le déroulement d'une course de F1.

#### Race Control (Contrôle de course)

*   **Valeurs manquantes :** Les colonnes `driver_number`, `flag`, `scope` et `sector` présentent un nombre important de valeurs manquantes. Il est crucial de comprendre la raison de ces manques (par exemple, tous les événements ne concernent pas un pilote spécifique).
*   **Catégories d'événements :** La catégorie la plus fréquente est "Flag", ce qui est logique pour des données de contrôle de course.

#### Stint (Relais)

*   **Valeurs manquantes :** Les colonnes `compound` et `tyre_age_at_start` présentent un faible pourcentage de valeurs manquantes.
*   **Type de pneu :** Le type de pneu le plus utilisé est le "SOFT", suivi du "MEDIUM".
*   **Âge des pneus :** L'âge moyen des pneus au début d'un relais est cohérent avec les stratégies de course habituelles.

#### Weather (Météo)

*   **Pas de valeurs manquantes :** Les données météorologiques sont complètes.
*   **Conditions météorologiques :** Les données indiquent une température moyenne de l'air de 23.8°C et une humidité relative de 53.6%, avec des variations typiques pour des conditions de course.

### Conclusion

Les données fournissent une base solide pour l'analyse des performances en Formule 1. Cependant, il est important de traiter les valeurs manquantes et les potentielles données aberrantes avant de procéder à des analyses plus approfondies. L'imputation des valeurs manquantes, en particulier pour les temps au tour et les intervalles, sera cruciale pour de nombreux objectifs de modélisation.


In [793]:
# Convertissons d'abord date_start en datetime avec le bon format
laps_df['date_start'] = pd.to_datetime(laps_df['date_start'], format='mixed')

# Maintenant calculons la date de début de chaque session
session_start_dates = sessions_df.merge(
    laps_df.groupby('session_key')['date_start'].min().reset_index(),
    on='session_key',
    how='left'
)
sessions_df['session_date_start'] = session_start_dates['date_start']

# Jointure des tables Drivers, Sessions et Laps
merged_df = pd.merge(drivers_df, sessions_df, on='session_key', how='inner')
merged_df = pd.merge(merged_df, laps_df, on=['driver_number', 'session_key'], how='inner')

# Conversion des colonnes de date en datetime64 avec gestion des erreurs
merged_df['date_start'] = pd.to_datetime(merged_df['date_start'], format='mixed', errors='coerce')
merged_df['session_date_start'] = pd.to_datetime(merged_df['session_date_start'], format='mixed', errors='coerce')

# Calcul de date_end
merged_df['date_end'] = merged_df['date_start'] + pd.to_timedelta(merged_df['lap_duration'], unit='s')

# Sélection des colonnes pertinentes avec ajout de session_date_start
final_df = merged_df[['session_key', 'meeting_key', 'circuit_key', 'year', 
                     'session_name', 'session_type', 'driver_number', 
                     'full_name', 'team_name', 'lap_number', 
                     'session_date_start', 
                     'date_start', 'date_end', 'duration_sector_1', 
                     'duration_sector_2', 'duration_sector_3', 'i1_speed', 
                     'i2_speed', 'is_pit_out_lap', 'lap_duration', 
                     'st_speed']].copy()

final_df.sort_values(by=['session_key', 'driver_number', 'lap_number'], inplace=True)

In [ ]:
# Affichage des premières lignes pour vérification
display(final_df.head())

In [ ]:
# Liste des colonnes à utiliser pour la moyenne glissante
rolling_columns = ['i1_speed', 'i2_speed', 'lap_duration', 'st_speed']

# Calcul des moyennes glissantes
for col in rolling_columns:
    # Gestion des valeurs manquantes
    temp_values = final_df[col].fillna(method='ffill')
    
    # Calcul de la moyenne glissante directement avec transform
    rolling_col = f'{col}_rolling_mean'
    final_df[rolling_col] = final_df.groupby(['session_key', 'driver_number'])[col] \
        .transform(lambda x: x.shift(1).rolling(window=5, min_periods=3).mean())

# Affichage d'un exemple de résultat pour vérifier les nouvelles colonnes
print("\nExemple de résultats avec les nouvelles colonnes :")
print(final_df[['session_key', 'driver_number', 'lap_number', 'i1_speed', 'i1_speed_rolling_mean',
      'i2_speed', 'i2_speed_rolling_mean', 'lap_duration', 'lap_duration_rolling_mean',
      'st_speed', 'st_speed_rolling_mean']].tail(10))

# Traitement de l'historique des race controls

In [ ]:
print("Ajout de l'historique des race controls...")

# Nombre maximum de race controls à conserver par tour
MAX_RC_EVENTS = 10

# Assurer que la colonne 'date' dans race_control_df est de type datetime
race_control_df['date'] = pd.to_datetime(race_control_df['date'], errors='coerce',utc=True)

# Préparation des race controls par session
rc_session_history = []

# Grouper les sessions par leurs dates de début et de fin
session_times = final_df.groupby('session_key').agg({
    'date_start': 'min',
    'date_end': 'max'
}).dropna()

for session_key, session_info in tqdm(session_times.iterrows(), desc="Traitement des sessions"):
    # Convertir les dates de début et de fin de session en Timestamp si elles ne le sont pas déjà
    session_start = pd.to_datetime(session_info['date_start'], utc=True)
    session_end = pd.to_datetime(session_info['date_end'], utc=True)

    # Obtenir tous les race controls de la session
    session_rcs = race_control_df[
        (race_control_df['date'] >= session_start) &
        (race_control_df['date'] <= session_end)
    ].sort_values('date', ascending=False)

    # Pour chaque tour de la session
    session_laps = final_df[final_df['session_key'] == session_key]
    for idx in session_laps.index:
        end_time = pd.to_datetime(session_laps.loc[idx, 'date_end'], utc=True)
        if pd.isna(end_time):
            continue

        #Obtenir les N derniers race controls jusqu'à la fin de ce tour, en excluant les 'Other'
        relevant_rcs = session_rcs[(session_rcs['date'] <= end_time) & (session_rcs['category'] != 'Other')].head(MAX_RC_EVENTS)
        row_data = {'index': idx}

        # Remplir les données
        rc_index = 1
        for _, rc in relevant_rcs.iterrows():
            time_diff = end_time - rc['date']
            row_data[f'session_rc_{rc_index}_category'] = rc['category']
            row_data[f'session_rc_{rc_index}_driver_number'] = rc['driver_number']
            row_data[f'session_rc_{rc_index}_flag'] = rc['flag']
            row_data[f'session_rc_{rc_index}_scope'] = rc['scope']
            row_data[f'session_rc_{rc_index}_sector'] = rc['sector']
            row_data[f'session_rc_{rc_index}_interval'] = time_diff.total_seconds()
            rc_index += 1

        rc_session_history.append(row_data)

# Création d'un DataFrame avec l'historique
rc_history_df = pd.DataFrame(rc_session_history).set_index('index')

# Ajout des colonnes d'historique au DataFrame principal
final_df_with_all = pd.concat([
    final_df,
    rc_history_df
], axis=1)

print("Terminé!")
display(final_df_with_all.head())

In [797]:
# Assurer les bons types de données et le tri
final_df_with_all['driver_number'] = final_df_with_all['driver_number'].astype('Int64')
final_df_with_all['lap_number'] = final_df_with_all['lap_number'].astype('Int64')

# Conversion des dates en datetime
final_df_with_all['date_start'] = pd.to_datetime(final_df_with_all['date_start'], utc=True)
final_df_with_all['date_end'] = pd.to_datetime(final_df_with_all['date_end'], utc=True)
final_df_with_all['session_date_start'] = pd.to_datetime(final_df_with_all['session_date_start'], utc=True)

# Traitement de position

In [ ]:
print("Traitement de position...")

position_df['date'] = pd.to_datetime(position_df['date'], format='ISO8601', utc=True)
position_df['driver_number'] = position_df['driver_number'].astype('Int64')
position_df['position'] = position_df['position'].astype('Int64')

# Initialisation de la colonne position avec des NaN
final_df_with_all['position'] = pd.NA

# Traitement session par session
for session_key, session_group in final_df_with_all.groupby('session_key'):
    # Ne garder que les lignes avec date_end non nulle
    valid_rows = session_group.dropna(subset=['date_end']).copy()  # Ajout de .copy()

    if len(valid_rows) == 0:
        continue

    session_start = session_group['session_date_start'].iloc[0]
    session_end = valid_rows['date_end'].max()

    # Filtrer les positions pour cette session uniquement
    session_positions = position_df[
        (position_df['date'] >= session_start) &
        (position_df['date'] <= session_end) &
        (position_df['driver_number'].isin(session_group['driver_number']))  # Ajout du filtre sur driver_number
    ].copy()  # Ajout de .copy()

    if len(session_positions) == 0:
        continue

    # Trier les données avant la jointure
    valid_rows = valid_rows.sort_values('date_end')
    session_positions = session_positions.sort_values('date')

    # Faire la jointure avec merge_asof
    result = pd.merge_asof(
        valid_rows,
        session_positions,
        left_on='date_end',
        right_on='date',
        by='driver_number',
        direction='backward',
        suffixes=('', '_y')
    )

    # Correction du décalage des positions
    for driver_number in result['driver_number'].unique():
      driver_mask = (result['driver_number'] == driver_number)
      driver_positions = result.loc[driver_mask, 'position_y']

      if not driver_positions.empty:
        # Décaler les positions d'un tour vers le bas
        shifted_positions = driver_positions.shift(1)

        # Mettre NaN pour la première position du pilote dans la session
        shifted_positions.iloc[0] = float('nan')

        # Mettre à jour les positions corrigées dans le DataFrame 'result'
        result.loc[driver_mask, 'position_y'] = shifted_positions

    # Mettre à jour les positions dans le DataFrame principal
    final_df_with_all.loc[valid_rows.index, 'position'] = result.set_index(valid_rows.index)['position_y']

final_df_with_all['position'] = final_df_with_all['position'].astype('Int64')

# Position au tour précédent
print("Traitement de position_last_lap...")
def get_previous_position(group):
    group = group.copy()
    # Création d'un shift standard
    group['position_last_lap'] = group['position'].shift(1)
    # Mise à NaN pour le premier tour
    group['position_last_lap'] = group.apply(
        lambda row: pd.NA if row['lap_number'] == 1 or pd.isna(row['position_last_lap'])
        else row['position_last_lap'],
        axis=1
    )
    return group['position_last_lap']

final_df_with_all['position_last_lap'] = final_df_with_all.groupby(
    ['session_key', 'driver_number'], 
    group_keys=False
).apply(get_previous_position).astype('Int64')

# Position 5 tours avant
print("Traitement de position_last_5_lap...")
def get_5_laps_ago_position(group):
    group = group.copy()
    # Création d'un shift standard
    group['position_last_5_lap'] = group['position'].shift(5)
    # Mise à NaN pour les   5 premiers tours
    group['position_last_5_lap'] = group.apply(
        lambda row: pd.NA if row['lap_number'] <= 5 or pd.isna(row['position_last_5_lap'])
        else row['position_last_5_lap'],
        axis=1
    )
    return group['position_last_5_lap']

final_df_with_all['position_last_5_lap'] = final_df_with_all.groupby(
    ['session_key', 'driver_number'], 
    group_keys=False
).apply(get_5_laps_ago_position).astype('Int64')

# Position 10 tours avant
print("Traitement de position_last_10_lap...")
def get_10_laps_ago_position(group):
    group = group.copy()
    # Création d'un shift standard
    group['position_last_10_lap'] = group['position'].shift(10)
    # Mise à NaN pour les 10 premiers tours
    group['position_last_10_lap'] = group.apply(
        lambda row: pd.NA if row['lap_number'] <= 10 or pd.isna(row['position_last_10_lap'])
        else row['position_last_10_lap'],
        axis=1
    )
    return group['position_last_10_lap']

final_df_with_all['position_last_10_lap'] = final_df_with_all.groupby(
    ['session_key', 'driver_number'], 
    group_keys=False
).apply(get_10_laps_ago_position).astype('Int64')

# Position au premier tour de la session
print("Traitement de position_at_start...")
first_lap_positions = final_df_with_all.groupby(['session_key', 'driver_number']).agg({
    'position': 'first',
    'lap_number': 'first'
}).reset_index()
first_lap_positions = first_lap_positions[first_lap_positions['lap_number'] == 1]
first_lap_positions = first_lap_positions[['session_key', 'driver_number', 'position']]
first_lap_positions.columns = ['session_key', 'driver_number', 'position_at_start']
first_lap_positions['position_at_start'] = first_lap_positions['position_at_start'].astype('Int64')

# Merge avec le DataFrame principal
final_df_with_all = final_df_with_all.merge(
    first_lap_positions,
    on=['session_key', 'driver_number'],
    how='left'
)

In [ ]:
# Affichage des résultats pour la session 7779 et le pilote 44
print("\nDonnées de position pour la session 7779 et le pilote 44:")
display(final_df_with_all[
    (final_df_with_all['session_key'] == 7779) & 
    (final_df_with_all['driver_number'] == 44)
][['session_key', 'driver_number', 'lap_number', 'date_start',
   'position', 'position_last_lap', 
   'position_last_10_lap', 'position_at_start']].sort_values('lap_number'))

In [ ]:
# Convertir les types et trier les données
position_df['date'] = pd.to_datetime(position_df['date'], format='ISO8601', utc=True)
position_df['driver_number'] = position_df['driver_number'].astype('Int64')
position_df['position'] = position_df['position'].astype('Int64')

# On récupère les dates de début et fin de la session 7779
session_dates = final_df_with_all[
    final_df_with_all['session_key'] == 7779
].agg({
    'session_date_start': 'min',
    'date_start': 'min',
    'date_end': 'max'
})

# Afficher les positions pour ce pilote pendant cette session
positions_during_session = position_df[
    (position_df['driver_number'] == 1) &
    (position_df['date'] >= session_dates['session_date_start']) &
    (position_df['date'] <= session_dates['date_end'])
].sort_values('date')

print(f"Positions pour le pilote 44 pendant la session 7779:")
print(f"Date de début de session: {session_dates['session_date_start']}")
print(f"Date de fin de session: {session_dates['date_end']}")
print("\nPositions:")
display(positions_during_session)

# Traitement des intervals

In [ ]:
import pandas as pd
import numpy as np

print("Traitement des intervals...")

# Assurer que les types de données sont corrects et que les dates sont au bon format
intervals_df['date'] = pd.to_datetime(intervals_df['date'], format='ISO8601', utc=True)
intervals_df['driver_number'] = intervals_df['driver_number'].astype('Int64')
intervals_df = intervals_df.dropna(subset=['date', 'driver_number'])

# Initialisation des colonnes interval et gap_to_leader avec des NaN
final_df_with_all['interval'] = pd.NA
final_df_with_all['gap_to_leader'] = pd.NA

# Liste des valeurs non numériques à gérer
non_numeric_values = ['+1 LAP', '1 L', '2 L', '+2 LAPS']

# Ajout des colonnes booléennes pour indiquer la présence de valeurs non numériques
final_df_with_all['interval_is_special_case'] = False
final_df_with_all['gap_is_special_case'] = False

# Traitement session par session
for session_key, session_group in final_df_with_all.groupby('session_key'):
    # Ne garder que les lignes avec date_end non nulle
    valid_rows = session_group.dropna(subset=['date_end']).copy()

    if len(valid_rows) == 0:
        continue

    session_start = session_group['session_date_start'].iloc[0]
    session_end = valid_rows['date_end'].max()

    # Filtrer les intervalles pour cette session uniquement
    session_intervals = intervals_df[
        (intervals_df['date'] >= session_start) &
        (intervals_df['date'] <= session_end) &
        (intervals_df['driver_number'].isin(session_group['driver_number']))
    ].copy()

    if len(session_intervals) == 0:
        continue

    # Trier les données avant la jointure
    valid_rows = valid_rows.sort_values('date_end')
    session_intervals = session_intervals.sort_values('date')

    # Faire la jointure avec merge_asof
    result = pd.merge_asof(
        valid_rows,
        session_intervals,
        left_on='date_end',
        right_on='date',
        by='driver_number',
        direction='backward',
        suffixes=('', '_y')
    )

    # Filtrer les résultats à plus d'une minute
    result['time_diff'] = (result['date_end'] - result['date']).dt.total_seconds()
    result.loc[result['time_diff'] > 60, ['interval_y', 'gap_to_leader_y']] = pd.NA

    # Correction du décalage des intervalles et des écarts au leader
    for driver_number in result['driver_number'].unique():
        driver_mask = (result['driver_number'] == driver_number)

        # Correction pour 'interval_y'
        driver_intervals = result.loc[driver_mask, 'interval_y']
        if not driver_intervals.empty:
            shifted_intervals = driver_intervals.shift(1)
            shifted_intervals.iloc[0] = pd.NA
            result.loc[driver_mask, 'interval_y'] = shifted_intervals

        # Correction pour 'gap_to_leader_y'
        driver_gaps = result.loc[driver_mask, 'gap_to_leader_y']
        if not driver_gaps.empty:
            shifted_gaps = driver_gaps.shift(1)
            shifted_gaps.iloc[0] = pd.NA
            result.loc[driver_mask, 'gap_to_leader_y'] = shifted_gaps

    # Identifier les cas spéciaux et mettre à jour les colonnes booléennes
    result['interval_is_special_case'] = result['interval_y'].isin(non_numeric_values)
    result['gap_is_special_case'] = result['gap_to_leader_y'].isin(non_numeric_values)

    # Remplacer les valeurs spéciales par NaN dans les colonnes 'interval_y' et 'gap_to_leader_y'
    result.loc[result['interval_is_special_case'], 'interval_y'] = np.nan
    result.loc[result['gap_is_special_case'], 'gap_to_leader_y'] = np.nan

    # Mettre à jour les colonnes dans le DataFrame principal
    final_df_with_all.loc[valid_rows.index, 'interval'] = result.set_index(valid_rows.index)['interval_y']
    final_df_with_all.loc[valid_rows.index, 'gap_to_leader'] = result.set_index(valid_rows.index)['gap_to_leader_y']
    final_df_with_all.loc[valid_rows.index, 'interval_is_special_case'] = result.set_index(valid_rows.index)['interval_is_special_case']
    final_df_with_all.loc[valid_rows.index, 'gap_is_special_case'] = result.set_index(valid_rows.index)['gap_is_special_case']

# Remplacer les NaT par NaN avant la conversion en float
final_df_with_all['interval'].fillna(value=np.nan, inplace=True)
final_df_with_all['gap_to_leader'].fillna(value=np.nan, inplace=True)

# Convertir les colonnes en types appropriés
final_df_with_all['interval'] = final_df_with_all['interval'].astype(float)
final_df_with_all['gap_to_leader'] = final_df_with_all['gap_to_leader'].astype(float)

# Interval au tour précédent
print("Traitement de interval_last_lap...")
def get_previous_interval(group):
    group = group.copy()
    group['interval_last_lap'] = group['interval'].shift(1)
    group['interval_last_lap'] = group.apply(
        lambda row: pd.NA if row['lap_number'] == 1 or pd.isna(row['interval_last_lap'])
        else row['interval_last_lap'],
        axis=1
    )
    return group['interval_last_lap']

final_df_with_all['interval_last_lap'] = final_df_with_all.groupby(
    ['session_key', 'driver_number'], 
    group_keys=False
).apply(get_previous_interval)

# Interval 5 tours avant
print("Traitement de interval_last_5_lap...")
def get_5_laps_ago_interval(group):
    group = group.copy()
    group['interval_last_5_lap'] = group['interval'].shift(5)
    group['interval_last_5_lap'] = group.apply(
        lambda row: pd.NA if row['lap_number'] <= 5 or pd.isna(row['interval_last_5_lap'])
        else row['interval_last_5_lap'],
        axis=1
    )
    return group['interval_last_5_lap']

final_df_with_all['interval_last_5_lap'] = final_df_with_all.groupby(
    ['session_key', 'driver_number'], 
    group_keys=False
).apply(get_5_laps_ago_interval)

# Interval 10 tours avant
print("Traitement de interval_last_10_lap...")
def get_10_laps_ago_interval(group):
    group = group.copy()
    group['interval_last_10_lap'] = group['interval'].shift(10)
    group['interval_last_10_lap'] = group.apply(
        lambda row: pd.NA if row['lap_number'] <= 10 or pd.isna(row['interval_last_10_lap'])
        else row['interval_last_10_lap'],
        axis=1
    )
    return group['interval_last_10_lap']

final_df_with_all['interval_last_10_lap'] = final_df_with_all.groupby(
    ['session_key', 'driver_number'], 
    group_keys=False
).apply(get_10_laps_ago_interval)

# Gap au tour précédent
print("Traitement de gap_to_leader_last_lap...")
def get_previous_gap(group):
    group = group.copy()
    group['gap_to_leader_last_lap'] = group['gap_to_leader'].shift(1)
    group['gap_to_leader_last_lap'] = group.apply(
        lambda row: pd.NA if row['lap_number'] == 1 or pd.isna(row['gap_to_leader_last_lap'])
        else row['gap_to_leader_last_lap'],
        axis=1
    )
    return group['gap_to_leader_last_lap']

final_df_with_all['gap_to_leader_last_lap'] = final_df_with_all.groupby(
    ['session_key', 'driver_number'], 
    group_keys=False
).apply(get_previous_gap)

# Gap 5 tours avant
print("Traitement de gap_to_leader_last_5_lap...")
def get_5_laps_ago_gap(group):
    group = group.copy()
    group['gap_to_leader_last_5_lap'] = group['gap_to_leader'].shift(5)
    group['gap_to_leader_last_5_lap'] = group.apply(
        lambda row: pd.NA if row['lap_number'] <= 5 or pd.isna(row['gap_to_leader_last_5_lap'])
        else row['gap_to_leader_last_5_lap'],
        axis=1
    )
    return group['gap_to_leader_last_5_lap']

final_df_with_all['gap_to_leader_last_5_lap'] = final_df_with_all.groupby(
    ['session_key', 'driver_number'], 
    group_keys=False
).apply(get_5_laps_ago_gap)

# Gap 10 tours avant
print("Traitement de gap_to_leader_last_10_lap...")
def get_10_laps_ago_gap(group):
    group = group.copy()
    group['gap_to_leader_last_10_lap'] = group['gap_to_leader'].shift(10)
    group['gap_to_leader_last_10_lap'] = group.apply(
        lambda row: pd.NA if row['lap_number'] <= 10 or pd.isna(row['gap_to_leader_last_10_lap'])
        else row['gap_to_leader_last_10_lap'],
        axis=1
    )
    return group['gap_to_leader_last_10_lap']

final_df_with_all['gap_to_leader_last_10_lap'] = final_df_with_all.groupby(
    ['session_key', 'driver_number'], 
    group_keys=False
).apply(get_10_laps_ago_gap)

# Colonnes booléennes pour les cas spéciaux des tours précédents
final_df_with_all['interval_last_lap_is_special_case'] = final_df_with_all['interval_last_lap'].isin(non_numeric_values)
final_df_with_all['interval_last_5_lap_is_special_case'] = final_df_with_all['interval_last_5_lap'].isin(non_numeric_values)
final_df_with_all['interval_last_10_lap_is_special_case'] = final_df_with_all['interval_last_10_lap'].isin(non_numeric_values)
final_df_with_all['gap_last_lap_is_special_case'] = final_df_with_all['gap_to_leader_last_lap'].isin(non_numeric_values)
final_df_with_all['gap_last_5_lap_is_special_case'] = final_df_with_all['gap_to_leader_last_5_lap'].isin(non_numeric_values)
final_df_with_all['gap_last_10_lap_is_special_case'] = final_df_with_all['gap_to_leader_last_10_lap'].isin(non_numeric_values)

# Remplacer les valeurs spéciales par NaN pour les tours précédents
final_df_with_all.loc[final_df_with_all['interval_last_lap_is_special_case'], 'interval_last_lap'] = np.nan
final_df_with_all.loc[final_df_with_all['interval_last_5_lap_is_special_case'], 'interval_last_5_lap'] = np.nan
final_df_with_all.loc[final_df_with_all['interval_last_10_lap_is_special_case'], 'interval_last_10_lap'] = np.nan
final_df_with_all.loc[final_df_with_all['gap_last_lap_is_special_case'], 'gap_to_leader_last_lap'] = np.nan
final_df_with_all.loc[final_df_with_all['gap_last_5_lap_is_special_case'], 'gap_to_leader_last_5_lap'] = np.nan
final_df_with_all.loc[final_df_with_all['gap_last_10_lap_is_special_case'], 'gap_to_leader_last_10_lap'] = np.nan

# Remplacer les NaT par NaN avant la conversion en float pour les tours précédents
final_df_with_all['interval_last_lap'].fillna(value=np.nan, inplace=True)
final_df_with_all['interval_last_5_lap'].fillna(value=np.nan, inplace=True)
final_df_with_all['interval_last_10_lap'].fillna(value=np.nan, inplace=True)
final_df_with_all['gap_to_leader_last_lap'].fillna(value=np.nan, inplace=True)
final_df_with_all['gap_to_leader_last_5_lap'].fillna(value=np.nan, inplace=True)
final_df_with_all['gap_to_leader_last_10_lap'].fillna(value=np.nan, inplace=True)

# Convertir les colonnes des tours précédents en float
final_df_with_all['interval_last_lap'] = final_df_with_all['interval_last_lap'].astype(float)
final_df_with_all['interval_last_5_lap'] = final_df_with_all['interval_last_5_lap'].astype(float)
final_df_with_all['interval_last_10_lap'] = final_df_with_all['interval_last_10_lap'].astype(float)
final_df_with_all['gap_to_leader_last_lap'] = final_df_with_all['gap_to_leader_last_lap'].astype(float)
final_df_with_all['gap_to_leader_last_5_lap'] = final_df_with_all['gap_to_leader_last_5_lap'].astype(float)
final_df_with_all['gap_to_leader_last_10_lap'] = final_df_with_all['gap_to_leader_last_10_lap'].astype(float)

print("Traitement des intervals terminé.")

In [ ]:
# Filtrer les données pour la session 9070 et le pilote 44
filtered_data = final_df_with_all[
    (final_df_with_all['session_key'] == 9070) & 
    (final_df_with_all['driver_number'] == 44)
]

# Afficher les colonnes pertinentes
columns_to_display = [
    'session_key', 'driver_number', 
    'interval', 'interval_last_lap', 'interval_last_5_lap', 'interval_last_10_lap', 'gap_to_leader', 
    'gap_to_leader_last_lap', 'gap_to_leader_last_5_lap', 'gap_to_leader_last_10_lap'
]

print(filtered_data[columns_to_display])

# Traitement de weather

In [ ]:
print(weather_df.dtypes)

In [ ]:
print("Traitement de la météo...")

# Assurer que les types de données sont corrects et que les dates sont au bon format
weather_df['date'] = pd.to_datetime(weather_df['date'], format='ISO8601', utc=True)
weather_df['rainfall'] = weather_df['rainfall'].astype('Int64')
weather_df['wind_direction'] = weather_df['wind_direction'].astype('Int64')

# Initialisation des colonnes de météo avec des NaN
weather_columns = ['air_temperature', 'humidity', 'pressure', 
                   'rainfall', 'track_temperature', 'wind_direction', 
                   'wind_speed']
for col in weather_columns:
    final_df_with_all[col] = pd.NA

# Traitement session par session
for session_key, session_group in final_df_with_all.groupby('session_key'):
    # Ne garder que les lignes avec date_end non nulle
    valid_rows = session_group.dropna(subset=['date_end']).copy()

    if len(valid_rows) == 0:
        continue

    session_start = session_group['session_date_start'].iloc[0]
    session_end = valid_rows['date_end'].max()

    # Filtrer les données météo pour cette session uniquement
    session_weather = weather_df[
        (weather_df['date'] >= session_start) &
        (weather_df['date'] <= session_end)
    ].copy()

    if len(session_weather) == 0:
        continue

    # Trier les données avant la jointure
    valid_rows = valid_rows.sort_values('date_end')
    session_weather = session_weather.sort_values('date')

    # Faire la jointure avec merge_asof
    result = pd.merge_asof(
        valid_rows,
        session_weather,
        left_on='date_end',
        right_on='date',
        direction='backward',
        suffixes=('', '_y')
    )

    # Correction du décalage pour toutes les colonnes météo d'un coup
    for col in weather_columns:
        # Décaler toutes les valeurs d'un rang
        result[f'{col}_y'] = result[f'{col}_y'].shift(20)
        # Mettre NA pour la première ligne
        result[f'{col}_y'].iloc[0] = pd.NA

    # Mise à jour du DataFrame principal
    for col in weather_columns:
        final_df_with_all.loc[valid_rows.index, col] = result.set_index(valid_rows.index)[f'{col}_y']

# Remplacer les NaT par NaN avant la conversion en float
for col in weather_columns:
    final_df_with_all[col].fillna(value=np.nan, inplace=True)

# Convertir les colonnes en types appropriés si nécessaire
final_df_with_all['air_temperature'] = final_df_with_all['air_temperature'].astype(float)
final_df_with_all['humidity'] = final_df_with_all['humidity'].astype(float)
final_df_with_all['pressure'] = final_df_with_all['pressure'].astype(float)
final_df_with_all['rainfall'] = final_df_with_all['rainfall'].astype('Int64')
final_df_with_all['track_temperature'] = final_df_with_all['track_temperature'].astype(float)
final_df_with_all['wind_direction'] = final_df_with_all['wind_direction'].astype('Int64')
final_df_with_all['wind_speed'] = final_df_with_all['wind_speed'].astype(float)

print("Traitement de la météo terminé.")

In [ ]:
# Filtrer les données
filtered_data = final_df_with_all[
    (final_df_with_all['session_key'] == 9515) & 
    (final_df_with_all['driver_number'] == 18)
]

# Afficher les colonnes pertinentes
columns_to_display = [
    'session_key', 'driver_number', 'date_start', 
    'air_temperature', 'humidity', 'pressure', 'rainfall', 'track_temperature', 
    'wind_direction', 'wind_speed'
]

print(filtered_data[columns_to_display])

# Traitement de pits

In [ ]:
# Traitement des pits (simple merge sur les clés)
print("Ajout des données pit...")
pit_merged = final_df_with_all.merge(
    pit_df[['driver_number', 'session_key', 'lap_number', 'pit_duration']],
    how='left',
    on=['driver_number', 'session_key', 'lap_number']
)

# Traitement des stints

In [ ]:
def process_stints_for_driver_session(driver_number, session_key, stint_df, laps_df):
    """
    Traite les stints pour un pilote et une session donnés.
    """
    # Filtrer les stints pour le pilote et la session donnés
    driver_stints = stint_df[
        (stint_df['driver_number'] == driver_number) &
        (stint_df['session_key'] == session_key)
    ].copy()

    # Trier les stints par lap_start
    driver_stints.sort_values(by='lap_start', inplace=True)

    # Filtrer les tours pour le pilote et la session donnés
    driver_laps = laps_df[
        (laps_df['driver_number'] == driver_number) &
        (laps_df['session_key'] == session_key)
    ].copy()

    # Initialiser les colonnes de stint
    driver_laps['stint_compound'] = None
    driver_laps['current_tyre_age'] = 0

    # Obtenir tous les composés uniques de la base de données
    all_compounds = stint_df['compound'].unique()
    
    # Initialiser les compteurs de stints à 0 pour tous les composés possibles
    stint_counts = {compound: 0 for compound in all_compounds}
    
    # Pour chaque composé, créer une colonne de compteur initialisée à 0
    for compound in all_compounds:
        driver_laps[f'{compound}_stint_count'] = 0

    # Itérer sur les stints
    for stint_index, stint_row in driver_stints.iterrows():
        compound = stint_row['compound']
        tyre_age_at_start = int(stint_row['tyre_age_at_start'])
        lap_start = int(stint_row['lap_start'])
        lap_end = int(stint_row['lap_end'])

        # Incrémenter le compteur pour ce composé
        stint_counts[compound] = stint_counts[compound] + 1
        current_stint_number = stint_counts[compound]

        # Mettre à jour les informations de stint pour les tours concernés
        lap_mask = (driver_laps['lap_number'] >= lap_start) & (driver_laps['lap_number'] <= lap_end)

        if lap_mask.any():
            driver_laps.loc[lap_mask, 'current_tyre_age'] = driver_laps.loc[lap_mask, 'lap_number'].apply(lambda x: tyre_age_at_start + (x - lap_start))
            driver_laps.loc[lap_mask, 'stint_compound'] = compound
            driver_laps.loc[lap_mask, f'{compound}_stint_count'] = current_stint_number

            # Mettre à jour tous les autres compteurs de stint pour ces tours
            for other_compound in all_compounds:
                if other_compound != compound:
                    previous_stint_count = stint_counts[other_compound]
                    driver_laps.loc[lap_mask, f'{other_compound}_stint_count'] = previous_stint_count

    # S'assurer que toutes les colonnes stint_count sont des entiers
    for compound in all_compounds:
        driver_laps[f'{compound}_stint_count'] = driver_laps[f'{compound}_stint_count'].astype(int)

    return driver_laps

def process_all_stints():
    """
    Traite les stints pour tous les pilotes et toutes les sessions.
    """
    all_laps = []

    # Itérer sur les pilotes et les sessions uniques
    for driver_number in tqdm(final_df_with_all['driver_number'].unique(), desc="Processing drivers"):
        for session_key in final_df_with_all[final_df_with_all['driver_number'] == driver_number]['session_key'].unique():
            processed_laps = process_stints_for_driver_session(
                driver_number, session_key, stint_df, final_df_with_all
            )
            all_laps.append(processed_laps)

    return pd.concat(all_laps)

# Traitement des stints avec la nouvelle stratégie
print("Ajout des données stint...")
stint_df_final = process_all_stints()

# Fusion finale
print("Fusion finale des données...")
final_df_with_all = stint_df_final

final_df_with_all = final_df_with_all.drop(columns=['date_end'])

print("Terminé!")

In [ ]:
final_df_with_all.head()

In [ ]:
# Afficher le nombre de lignes
nombre_de_lignes = final_df_with_all.shape[0]
print(f"Nombre de lignes : {nombre_de_lignes}")

# Afficher le nombre de colonnes
nombre_de_colonnes = final_df_with_all.shape[1]
print(f"Nombre de colonnes : {nombre_de_colonnes}")

In [ ]:
for col in final_df_with_all.columns:
    print(col)

In [ ]:
# Obtenir le chemin du bureau
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")

# Créer le nom du fichier
filename = f"f1_data_processed.csv"

# Chemin complet
full_path = os.path.join(desktop_path, filename)

# Sauvegarder en CSV
final_df_with_all.to_csv(full_path, index=True)

print(f"Fichier sauvegardé sous : {full_path}")

| Nom de la Colonne                 | Description                                                                                               | Type de Données | Unité/Format                 | Remarques                                                                                                |
| :--------------------------------- | :-------------------------------------------------------------------------------------------------------- | :-------------- | :--------------------------- | :------------------------------------------------------------------------------------------------------- |
| `index`                            | Index du DataFrame                                                                                   | `Int64`         |                              |                                                                                                          |
| `session_key`                      | Clé unique identifiant la session (ex: 9515).                                                         | `Int64`         |                              | Clé primaire de la table `sessions`                                                                     |
| `meeting_key`                      | Clé unique identifiant le meeting (l'événement de course).                                             | `Int64`         |                              |                                                                                                          |
| `circuit_key`                      | Clé unique identifiant le circuit.                                                                    | `Int64`         |                              |                                                                                                          |
| `year`                             | Année de la session.                                                                                   | `Int64`         | `YYYY`                       |                                                                                                          |
| `session_name`                     | Nom de la session (ex: "Practice 1", "Qualifying", "Race").                                              | `object`        |                              |                                                                                                          |
| `session_type`                     | Type de session (ex: "Practice", "Qualifying", "Race").                                                 | `object`        |                              |                                                                                                          |
| `driver_number`                    | Numéro unique du pilote.                                                                               | `Int64`         |                              | Clé primaire de la table `drivers`                                                                      |
| `full_name`                        | Nom complet du pilote.                                                                                | `object`        |                              |                                                                                                          |
| `team_name`                        | Nom de l'équipe du pilote.                                                                            | `object`        |                              | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `lap_number`                       | Numéro du tour dans la session.                                                                       | `Int64`         |                              |                                                                                                          |
| `session_date_start`               | Date et heure de début de la session.                                                                 | `datetime64[ns, UTC]` | `YYYY-MM-DD HH:MM:SS`        |                                                                                                          |
| `date_start`                       | Date et heure de début du tour.                                                                      | `datetime64[ns, UTC]` | `YYYY-MM-DD HH:MM:SS`        |                                                                                                          |
| `duration_sector_1`                | Durée du secteur 1 du tour.                                                                             | `float64`       | secondes                    | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `duration_sector_2`                | Durée du secteur 2 du tour.                                                                             | `float64`       | secondes                    | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `duration_sector_3`                | Durée du secteur 3 du tour.                                                                             | `float64`       | secondes                    | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `i1_speed`                         | Vitesse du pilote au point intermédiaire 1 (Intermédiaire 1).                                            | `float64`       | km/h                      | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `i2_speed`                         | Vitesse du pilote au point intermédiaire 2 (Intermédiaire 2).                                            | `float64`       | km/h                      | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `is_pit_out_lap`                   | Indique si le tour est un tour de sortie des stands (True) ou non (False).                                | `object`        | `True`/`False`               |                                                                                                          |
| `lap_duration`                     | Durée totale du tour.                                                                                | `float64`       | secondes                    | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `st_speed`                         | Vitesse du pilote au speed trap.                                                                      | `float64`       | km/h                      | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `i1_speed_rolling_mean`            | Moyenne glissante de la vitesse au point intermédiaire 1 sur les 5 derniers tours (minimum 3 tours).     | `float64`       | km/h                      | Peut contenir des valeurs manquantes (`NaN`) au début de la session ou après un arrêt aux stands          |
| `i2_speed_rolling_mean`            | Moyenne glissante de la vitesse au point intermédiaire 2 sur les 5 derniers tours (minimum 3 tours).     | `float64`       | km/h                      | Peut contenir des valeurs manquantes (`NaN`) au début de la session ou après un arrêt aux stands          |
| `lap_duration_rolling_mean`        | Moyenne glissante de la durée du tour sur les 5 derniers tours (minimum 3 tours).                      | `float64`       | secondes                    | Peut contenir des valeurs manquantes (`NaN`) au début de la session ou après un arrêt aux stands          |
| `st_speed_rolling_mean`            | Moyenne glissante de la vitesse au speed trap sur les 5 derniers tours (minimum 3 tours).               | `float64`       | km/h                      | Peut contenir des valeurs manquantes (`NaN`) au début de la session ou après un arrêt aux stands          |
| `session_rc_N_category`           | Catégorie du Nième événement de race control le plus récent avant la fin du tour (N entre 1 et 10).       | `object`        |                              | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `session_rc_N_driver_number`      | Numéro du pilote impliqué dans le Nième événement de race control (N entre 1 et 10).                   | `float64`       |                              | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `session_rc_N_flag`               | Drapeau associé au Nième événement de race control (ex: "GREEN", "RED", "YELLOW") (N entre 1 et 10).    | `object`        |                              | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `session_rc_N_scope`              | Portée du Nième événement de race control (ex: "TRACK", "SECTOR") (N entre 1 et 10).                    | `object`        |                              | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `session_rc_N_sector`             | Secteur concerné par le Nième événement de race control (N entre 1 et 10).                             | `float64`       |                              | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `session_rc_N_interval`           | Intervalle de temps (en secondes) entre la fin du tour et le Nième événement de race control (N entre 1 et 10). | `float64`       | secondes                    | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `position`                        | Position du pilote à la fin du tour.                                                                 | `Int64`         |                              | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `position_last_lap`               | Position du pilote à la fin du tour précédent.                                                       | `Int64`         |                              | `NaN` pour le premier tour, ou si la position du tour précédent est manquante.                             |
| `position_last_5_lap`             | Position du pilote à la fin du tour il y a 5 tours.                                                    | `Int64`         |                              | `NaN` pour les 5 premiers tours, ou si la position d'il y a 5 tours est manquante.                         |
| `position_last_10_lap`            | Position du pilote à la fin du tour il y a 10 tours.                                                   | `Int64`         |                              | `NaN` pour les 10 premiers tours, ou si la position d'il y a 10 tours est manquante.                       |
| `position_at_start`               | Position du pilote au début de la session (au tour 1).                                                 | `Int64`         |                              | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `interval`                        | Intervalle de temps avec le pilote précédent à la fin du tour.                                         | `float64`       | secondes                    | Peut contenir des valeurs manquantes (`NaN`).                                                              |
| `gap_to_leader`                   | Écart de temps avec le leader à la fin du tour.                                                        | `float64`       | secondes                    | Peut contenir des valeurs manquantes (`NaN`).                                                              |
| `interval_is_special_case`        | Indique si la valeur de `interval` est un cas spécial (ex: "+1 LAP", "1 L").                            | `boolean`       | `True`/`False`               |                                                                                                          |
| `gap_is_special_case`             | Indique si la valeur de `gap_to_leader` est un cas spécial.                                           | `boolean`       | `True`/`False`               |                                                                                                          |
| `interval_last_lap`               | Intervalle de temps avec le pilote précédent à la fin du tour précédent.                                | `float64`       | secondes                    | `NaN` pour le premier tour, ou si l'intervalle du tour précédent est manquant ou un cas spécial.           |
| `interval_last_5_lap`             | Intervalle de temps avec le pilote précédent à la fin du tour il y a 5 tours.                           | `float64`       | secondes                    | `NaN` pour les 5 premiers tours, ou si l'intervalle d'il y a 5 tours est manquant ou un cas spécial.       |
| `interval_last_10_lap`            | Intervalle de temps avec le pilote précédent à la fin du tour il y a 10 tours.                          | `float64`       | secondes                    | `NaN` pour les 10 premiers tours, ou si l'intervalle d'il y a 10 tours est manquant ou un cas spécial.     |
| `gap_to_leader_last_lap`          | Écart de temps avec le leader à la fin du tour précédent.                                                | `float64`       | secondes                    | `NaN` pour le premier tour, ou si l'écart du tour précédent est manquant ou un cas spécial.                 |
| `gap_to_leader_last_5_lap`        | Écart de temps avec le leader à la fin du tour il y a 5 tours.                                           | `float64`       | secondes                    | `NaN` pour les 5 premiers tours, ou si l'écart d'il y a 5 tours est manquant ou un cas spécial.             |
| `gap_to_leader_last_10_lap`       | Écart de temps avec le leader à la fin du tour il y a 10 tours.                                          | `float64`       | secondes                    | `NaN` pour les 10 premiers tours, ou si l'écart d'il y a 10 tours est manquant ou un cas spécial.           |
| `interval_last_lap_is_special_case` | Indique si la valeur de `interval_last_lap` est un cas spécial.                                        | `boolean`       | `True`/`False`               |                                                                                                          |
| `interval_last_5_lap_is_special_case` | Indique si la valeur de `interval_last_5_lap` est un cas spécial.                                       | `boolean`       | `True`/`False`               |                                                                                                          |
| `interval_last_10_lap_is_special_case`| Indique si la valeur de `interval_last_10_lap` est un cas spécial.                                      | `boolean`       | `True`/`False`               |                                                                                                          |
| `gap_last_lap_is_special_case`      | Indique si la valeur de `gap_to_leader_last_lap` est un cas spécial.                                   | `boolean`       | `True`/`False`               |                                                                                                          |
| `gap_last_5_lap_is_special_case`     | Indique si la valeur de `gap_to_leader_last_5_lap` est un cas spécial.                                  | `boolean`       | `True`/`False`               |                                                                                                          |
| `gap_last_10_lap_is_special_case`    | Indique si la valeur de `gap_to_leader_last_10_lap` est un cas spécial.                                 | `boolean`       | `True`/`False`               |                                                                                                          |
| `air_temperature`                  | Température de l'air.                                                                                 | `float64`       | °C                       | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `humidity`                         | Humidité relative.                                                                                     | `float64`       | %                        | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `pressure`                         | Pression atmosphérique.                                                                                | `float64`       | mbar                     | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `rainfall`                         | Présence de pluie (1 = pluie, 0 = pas de pluie).                                                      | `Int64`         | 0 ou 1                   | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `track_temperature`                | Température de la piste.                                                                              | `float64`       | °C                       | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `wind_direction`                   | Direction du vent.                                                                                    | `Int64`         | degrés (0-360)            | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `wind_speed`                       | Vitesse du vent.                                                                                      | `float64`       | km/h                      | Peut contenir des valeurs manquantes (`NaN`)                                                               |
| `pit_duration`                     | Durée de l'arrêt aux stands pour ce tour.                                                              | `float64`       | secondes                    | `NaN` si le pilote n'est pas rentré aux stands.                                                         |
|`stint_compound` | Type de gomme de pneu utilisée dans le stint actuel | `object` |  |  Peut contenir des valeurs manquantes (`NaN`) |
|`current_tyre_age` | Nombre de tours effectués avec le pneu actuel. | `int64` | tours | Zéro pour un pneu neuf |
|`SOFT_stint_count` | Compteur de stints effectués avec des pneus SOFT. | `int64` |  |  |
|`MEDIUM_stint_count` | Compteur de stints effectués avec des pneus MEDIUM. | `int64` |  |  |
|`UNKNOWN_stint_count` | Compteur de stints effectués avec des pneus de type UNKNOWN. | `int64` |  |  |
|`INTERMEDIATE_stint_count` | Compteur de stints effectués avec des pneus INTERMEDIATE. | `int64` |  |  |
|`HARD_stint_count` | Compteur de stints effectués avec des pneus HARD. | `int64` |  |  |
|`WET_stint_count` | Compteur de stints effectués avec des pneus WET. | `int64` |  |  |

**Remarques importantes:**

*   Les colonnes `Int64` avec un "I" majuscule indiquent des entiers qui peuvent contenir des valeurs manquantes (`NaN`), contrairement aux `int64` classiques.
*   Les colonnes avec `_rolling_mean` sont des moyennes glissantes calculées sur les 5 derniers tours (avec un minimum de 3 tours pour que la moyenne soit calculée). Elles seront `NaN` pour les premiers tours de la session et potentiellement après un arrêt aux stands.
*   Les colonnes `session_rc_N` (où N est un nombre de 1 à 10) contiennent des informations sur les 10 derniers événements de "race control" (ex: drapeaux, safety car) avant la fin du tour.
*   Les colonnes liées aux intervalles (`interval` et `gap_to_leader`) peuvent contenir des cas spéciaux comme "+1 LAP" ou "1 L" quand un pilote a un tour de retard. Ces cas sont indiqués par les colonnes `_is_special_case`.
*   Certaines colonnes, comme `team_name`, `duration_sector_1`, `i1_speed`, `i2_speed`, `lap_duration`, `st_speed`, peuvent contenir un nombre significatif de valeurs manquantes (`NaN`).
* La colonne `is_pit_out_lap` est de type `object` et non booléen. Il faudra la convertir si besoin.